# Datamine PANELEST Process Example

This notebook demonstrates how to configure and run the **`panelest`** process wrapper in `dmstudio`.

## Process Description

## PANELEST

Estimate grade and variance of 2D and 3D panels.

Panels are defined either as a set of strings from file PERIM, or as a set of 2D or 3D [discretisation](<../STUDIO_RM/Grade%20Estimation%20Cell%20Discretisation.md>) points from file DISPTIN. The interpolation methods available are nearest neighbour, inverse power of distance or Kriging.

In particular the process allows you to estimate a grade and a Kriged variance for:

* Any perimeter, without the need to create a block model.

* A subset of cells from a block model.

See also [PANELK](<panelk.md>) Process.

### Using Strings

The **PERIM** file may contain one or more strings. If the strings are not closed the process will close them to form the panel. You must ensure that a string does not intersect itself as this will lead to incorrect results. String intersections are not checked by the process. There is a limit of 5000 points in any one string.

The strings must be planar and must lie in one of the three orthogonal planes. If a string does not satisfy both these criteria then a warning will be displayed and that string will not be evaluated. Although the strings must be in one of the three orthogonal planes, they do not all have to be in the same plane.

You can define independent plus and minus projection distances using the parameters DPLUS and DMINUS in order to create a volume. If either of these values are non zero then the panel will be considered as a 3D panel; otherwise it is a 2D panel.

If a panel is 2D then all sample data will be projected onto the plane of the panel even if all three coordinates (X,Y,Z) are specified for the sample data. Hence the evaluation will be in 2D. If you are kriging a 2D panel then you must take care that projecting data onto the plane of the panel does not lead to coincident samples. The process will terminate with an error if there are coincident samples and kriging is used.

Each panel is represented by a set of 2D or 3D discretisation points. You can define the spacing between discretisation points independently in each of the three dimensions using the parameters XDSPACE, YDSPACE and ZDSPACE. Alternatively if you set these parameters to zero the process will calculate a suitable spacing between points. It does this by calculating the maximum extent of the panel in the two dimensions of its plane, and then calculating the corresponding area. This area is equivalent to a rectangle surrounding the panel. Taking the square root of the area gives the side of a square with the same area. The distance between points is then calculated by dividing the length of the side of this square by 10.5. In other words if the original panel were square it would contain 11 x 11 (=121) discretisation points. XDSPACE, YDSPACE and ZDSPACE are originally set to zero then they will all be assigned the same value, as described above, so that the discretisation points are regularly distributed.

A set of discretisation points are generated within the rectangle surrounding the panel, and then the points lying within the panel are selected. If it is a 3D panel then points are also generated in the third dimension, limited by the DPLUS and DMINUS values. The total number of discretisation points lying within the panel is calculated and is compared to the minimum number of points as defined by parameter MINDISC. If the number of points is less than MINDISC then the spacing in each dimension is multiplied by 0.8, a new set of points is generated, and the total is again compared to MINDISC. This procedure is repeated until the total exceeds MINDISC. If insufficient points are found after 10 iterations then a warning message is displayed and the panel is not estimated. If this happens then reduce the spacing, and/or MINDISC, and try again.

If parameter PRINT=2 then the coordinates of the discretisation points will be displayed in the Output window. If PRINT=2 and ECHO=1 then the points will also be written to the print file. If the points are required for further analysis within Datamine, they can be imported directly from the print file.

### Using Points

Instead of generating the discretisation points from strings, as defined above, you can input the discretisation points directly using the DISPTIN file. In this case the parameters XDSPACE, YDSPACE, ZDSPACE, MINDISC, DPLUS and DMINUS are ignored. One method of creating the points could be to use the [TRIFIL](<trifil.md>) process to create a set of cells within an enclosed wireframe or below a DTM. The centre points of the model cells would then be the discretisation points, and so the panel would represent the volume within the wireframe or below the DTM. The output model from **TRIFIL** can be input directly to PANELEST as the DISPTIN file, specifying the **XPT** field as XC, YPT as YC and ZPT as ZC. It is recommended that if you use **TRIFIL** to create the discretisation points then you should not allow any subcelling, so that the points are on a regular grid.

The discretisation points could also be the cell centres of an existing model for which you have already interpolated grade. By selecting a subset of the cells, or the total model, you can estimate a single kriged grade and in particular a single kriged variance for that subset of cells. If the model contains subcells then it would be best to regularise the model first, so that the points are on a regular grid.

### Sample Selection

The MINNUM and MAXNUM parameters allow you to select the minimum and maximum number of samples to be used for making each estimate. For Nearest Neighbour and Inverse Power of Distance there is no limit on MAXNUM. However for kriging MAXNUM must not exceed 1399.

If less than MINNUM samples are found then the panel will not be estimated. If there are more than MAXNUM, then the number is reduced until only MAXNUM remain. This is done by calculating the distance of each sample from the nearest discretisation point. The samples are then sorted according to this distance, and the MAXNUM samples with the smallest distances are selected.

If panels are defined using the PERIM file, then setting parameter INSIDE=1 will ensure that only samples which lie within the panel are used for the grade estimate; if INSIDE=0 then all samples will be considered. If panels are defined using the DISPTIN file, then the parameter **INSIDE** is ignored.

### Grade Estimation Method

The IMETHOD parameter allows you to select the estimation method.

See [Nearest Neighbour](<../STUDIO_RM/Grade%20Estimation%20Nearest%20Neighbour.md>), [kriging](<../STUDIO_RM/Grade%20Estimation%20Kriging.md>) and [Inverse Power of Distance](<../STUDIO_RM/Grade%20Estimation%20Inverse%20Power%20of%20Distance.md>).

1. Nearest Neighbour.

For each discretisation point the nearest sample is found. The panel estimate is then the arithmetic mean over all discretisation points. The reported variance is the classical statistical variance of all selected samples.

2. Inverse Power of Distance.

Each discretisation point is estimated using inverse power of distance, where the power is defined by parameter POWER. The panel estimate is then the arithmetic mean over all discretisation points. The reported variance is the classical statistical variance of all selected samples.

3. Kriging.

If parameter **LOG** =0 then normal kriging is used. If **LOG** =1 then log kriging is used, and the following conditions apply:
\- General Case method
\- a maximum of 3 iterations
\- convergence tolerance is set at 0.01
\- the kriged estimate is used as the mean value for the lognormal variance calculation.

If Kriging is selected then the reported variance is the Kriged variance. Note that this is different from Nearest Neighbour and Inverse Power of Distance which report the classical statistical variance of the selected samples.

### Anisotropy

If Kriging is selected then [anisotropy](<../STUDIO_RM/Dynamic%20Anisotropy%20-%20Introduction.md>) is defined by the parameters of the [variogram model](<../STUDIO_RM/Grade%20Estimation%20Variograms.md>).

For Nearest Neighbour or Inverse Power of Distance anisotropy is defined using the ANANGLEn, ANAXISn and ANDISTn parameters.

### Totals and Averages

If parameter **TOTAL** =1 and more than one panel has been estimated, then the total area and/or volume and the average grade, weighted by area or volume will be reported in the Output window and saved to the OUT file. The total variance will only be reported if kriging has been used. It is calculated as the weighted average of the individual kriged variances, weighted by the square of the corresponding area or volume. In calculating the total variance it is therefore assumed that the estimates and variances of the individual panels are independent of one another. This will often be a reasonable assumption if the panels are large.

If the panels are defined using the **DISPTIN** file then the volume of each panel must be estimated. This is done by finding the minimum distance between [discretisation](<../STUDIO_RM/Grade%20Estimation%20Cell%20Discretisation.md>) points in each of the X, Y and Z directions and calculating the corresponding volume of influence of a point. The volume of the panel is then the volume of influence of a point multiplied by the number of points. This will give an accurate estimate of volume if the points are on a regular grid, but not otherwise.

### Average Sample-Panel Variogram Value

As described above, if kriging is used then the average variogram value of each selected sample with the panel is written to the sample output file **SAMPOUT**. Sometimes these values are required for further processing and the kriged estimate and variance are not required. The run time is then very much quicker if parameter **VGONLY** is set to 1, so that the kriged estimate and variance are not calculated. Another advantage is that the maximum number of samples per panel in the **SAMPOUT** file is increased from 1,399 to 50,000.

### Timing

The execution speed of the process increases as the number of samples increases and also as the number of discretisation points increases. Also Kriging is significantly slower than Inverse Power of Distance.

### Results

The results for each panel are displayed in the **Output** window. If [kriging](<../STUDIO_RM/Grade%20Estimation%20Kriging.md>) has been selected and PRINT=1 then the panel F value (the variance of a point in the panel) and the Lagrange multiplier are also displayed. There are two output files, both of which are optional, examples of which are also shown below. The OUT file contains a single record for each panel, summarising the results. The SAMPOUT file shows the weight assigned to each sample for each panel. If kriging is used the **SAMPOUT** file also includes the average variogram value of the sample with the panel.

### Input Files:

* **in** (Undefined):
  Input sample data file. This must contain the fields **X , Y** , and **VALUE**. Note: A
  **Z** field is also required for 3D panels.
  Required=Yes

* **vmodparm** (Variogram - Model):
  Variogram model parameter file.
  Required=No

* **perim** (String):
  The input string file. This must contain the 5 fields **PANEL , PTN , XP , YP , ZP** .
  The strings may be open or closed, but must be planar and must lie in one of the three
  orthogonal planes. Either a PERIM file or a DISPTIN file must be specified.
  Required=No

* **disptin** (Point Data):
  The input file containing discretisation points. This must include the 3 fields **XPT ,
  YPT , ZPT** A fourth field **PANEL** is optional, and is used to identify different sets
  of discretisation points representing different panels. Either a **PERIM** file or a
  **DISPTIN** file must be specified.
  Required=No

### Output Files:

* **out** (Undefined):
  The output results file, containing a record for each panel estimated. The fields
  include the panel identifier, the estimated grade, the variance, and other associated
  information.
  Required=No

* **sampout** (Undefined):
  The sample output file. This will contain the samples used to estimate each panel, and
  the weight assigned to each sample.
  Required=No

### Fields:

* **x** (Numeric : IN):
  X coordinate of sample data in the IN file.
  Default=X
  Required=Yes

* **y** (Numeric : IN):
  Y coordinate of sample data in the IN file.
  Default=Y
  Required=Yes

* **z** (Numeric : IN):
  Z coordinate of sample data in the IN file.
  Default=Z
  Required=No

* **value** (Numeric : IN):
  Field to be estimated in the IN file.
  Default=Undefined
  Required=Yes

* **panel** (Numeric : PERIM,DISPTIN):
  Panel identifier. This is a numeric or alpha field (max 40 characters) in the **PERIM**
  or **DISPTIN** file. If a perimeter file is used then the **PVALUE** field may be used.
  Default=PVALUE
  Required=No

* **xpt** (Numeric : DISPTIN):
  X coordinate of discretisation points in the **DISPTIN** file.
  Default=Undefined
  Required=No

* **ypt** (Numeric : DISPTIN):
  Y coordinate of discretisation points in the **DISPTIN** file.
  Default=Undefined
  Required=No

* **zpt** (Numeric : DISPTIN):
  Z coordinate of discretisation points in the **DISPTIN** file.
  Default=Undefined
  Required=No

### Parameters:

* **minnum**:
  Minimum number of samples for panel to be estimated.(1).
  Range=1,+
  Values=Undefined
  Default=1
  Required=No

* **maxnum**:
  Maximum number of samples for panel to be estimated. If Kriging is selected then the
  maximum cannot exceed 1399.
  Range=1,+
  Values=Undefined
  Default=100
  Required=No

* **inside**:
  If set to 1 samples must lie inside panel. This only applies if a **PERIM** file has
  been specified.
  Range=0,1
  Values=0,1
  Default=1
  Required=No

* **xdspace**:
  The distance between discretisation points in X. If set to zero then a suitable spacing
  will be generated automatically.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **ydspace**:
  The distance between discretisation points in Y. If set to zero then a suitable spacing
  will be generated automatically.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **zdspace**:
  The distance between discretisation points in Z. If set to zero then a suitable spacing
  will be generated automatically.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **mindisc**:
  Minimum number of discretisation points in panel before it can be estimated.
  Range=1,+
  Values=Undefined
  Default=50
  Required=No

* **dplus**:
  Perimeter projection distance measured in the increasing direction of the perpendicular
  axis.
  Range=0,+
  Values=Undefined
  Default=0
  Required=No

* **dminus**:
  Perimeter projection distance measured in the decreasing direction of the perpendicular
  axis.
  Range=0,+
  Values=Undefined
  Default=0
  Required=No

* **imethod**:
  Interpolation method: 1 - nearest neighbour 2 - inverse power of distance 3 - ordinary
  kriging
  Range=1,3
  Values=1,2,3
  Default=3
  Required=No

* **vmodnum**:
  Variogram model reference number as defined by the **VREFNUM** field in the
  **VMODPARM** file. (1).
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **log**:
  Flag to indicate whether log kriging is selected. 0 = ordinary kriging 1 = log kriging
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **power**:
  Power for inverse power of distance method.
  Range=Undefined
  Values=Undefined
  Default=2
  Required=No

* **total**:
  If **TOTAL** is set to 1 then values for the total volume and area over all panels will
  be reported and saved to the **OUT** file. If kriging is selected then the average
  variance will be calculated as the weighted average of the individual variances,
  weighted by the square of the area or volume.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **vgonly**:
  Flag controlling estimation (0). Options: **0**: \- Calculate estimated grade and
  variance for each panel.; **1**: \- Only create the sample output file. Do not calculate
  estimated grade and variance.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **anangle1**:
  First rotation angle for defining anisotropy when nearest neighbour or inverse power of
  distance methods are selected ie **IMETHOD** = 1 or 2.
  Range=-360, 360
  Values=Undefined
  Default=0
  Required=No

* **anaxis1**:
  First rotation axis for defining anisotropy when nearest neighbour or inverse power of
  distance methods are selected ie **IMETHOD** = 1 or 2. This parameter has a value 1 for
  rotation about the X axis, 2 for rotation about the Y axis, and 3 for rotation about the
  Z axis. If it is set to 0 then there will be no rotation, irrespective of the value of

## **ANANGLE1**.

  Range=0,3
  Values=0,1,2,3
  Default=3
  Required=No

* **anangle2**:
  Second rotation angle for defining anisotropy when nearest neighbour or inverse power
  of distance methods are selected, that is,**IMETHOD** = 1 or 2.
  Range=-360, 360
  Values=Undefined
  Default=0
  Required=No

* **anaxis2**:
  Second rotation axis for defining anisotropy when nearest neighbour or inverse power of
  distance methods are selected ie **IMETHOD** = 1 or 2. This parameter has a value 1 for
  rotation about the X axis, 2 for rotation about the Y axis, and 3 for rotation about the
  Z axis. If it is set to 0 then there will be no rotation, irrespective of the value of

## **ANANGLE2**.

  Range=0,3
  Values=0,1,2,3
  Default=1
  Required=No

* **anangle3**:
  Third rotation angle for defining anisotropy when nearest neighbour or inverse power of
  distance methods are selected, that is **IMETHOD** = 1 or 2.
  Range=-360, 360
  Values=Undefined
  Default=0
  Required=No

* **anaxis3**:
  Third rotation axis for defining anisotropy when nearest neighbour or inverse power of
  distance methods are selected ie **IMETHOD** = 1 or 2. This parameter has a value 1 for
  rotation about the X axis, 2 for rotation about the Y axis, and 3 for rotation about the
  Z axis. If it is set to 0 then there will be no rotation, irrespective of the value of

## **ANANGLE3**.

  Range=0,3
  Values=0,1,2,3
  Default=3
  Required=No

* **andist1**:
  Anisotropy distance measured along rotated X axis, when nearest neighbour or inverse
  power of distance methods are selected ie **IMETHOD** = 1 or 2. This corresponds to the
  range of influence in that direction.
  Range=0.0001,+
  Values=Undefined
  Default=1
  Required=No

* **andist2**:
  Anisotropy distance measured along rotated Y axis, when nearest neighbour or inverse
  power of distance methods are selected ie **IMETHOD** = 1 or 2. This corresponds to the
  range of influence in that direction.
  Range=0.0001,+
  Values=Undefined
  Default=1
  Required=No

* **andist3**:
  Anisotropy distance measured along rotated Z axis, when nearest neighbour or inverse
  power of distance methods are selected, that is **IMETHOD** = 1 or 2. This corresponds
  to the range of influence in that direction.
  Range=0.0001,+
  Values=Undefined
  Default=1
  Required=No

* **print**:
  Output control flag (1). Options: **0**: Minimum output. This includes a summary of the
  input data and the results.; **1**: As 0 plus Lagrange multiplier and panel F value;
  **2**: As 1 plus a listing of discretisation points.
  Range=0,2
  Values=0,1,2
  Default=1
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('panelest')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute panelest
print("Running panelest...")
dm_cmd.panelest(
    in_i='t_assays',  # required
    sampout_o='t_panelest_out',  # required
    x_f='X',  # required
    y_f='Y',  # required
    z_f='Z',  # required
    value_f='optional',  # required
    # vmodparm_i='t_vpar',  # optional
    # perim_i='optional',  # optional
    # disptin_i='optional',  # optional
    # out_o='t_panelest_out',  # optional
    # panel_f='optional',  # optional
    # xpt_f='optional',  # optional
    # ypt_f='optional',  # optional
    # zpt_f='optional',  # optional
    # minnum_p=1,  # optional
    # maxnum_p=100,  # optional
    # inside_p=1,  # optional
    # xdspace_p=0,  # optional
    # ydspace_p=0,  # optional
    # zdspace_p=0,  # optional
    # mindisc_p=50,  # optional
    # dplus_p=0,  # optional
    # dminus_p=0,  # optional
    # imethod_p=3,  # optional
    # vmodnum_p=1,  # optional
    # log_p=0,  # optional
    # power_p=2,  # optional
    # total_p=0,  # optional
    # vgonly_p=0,  # optional
    # anangles_f=['optional'],  # optional
    # anaxiss_f=['optional'],  # optional
    # andists_f=['optional'],  # optional
    # print_p=1,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("panelest execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_panelest_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")